# Week 2 — Build and Debug an Agent Tool

In this notebook, you will:

1. compare Agents with different configurations and inspect what changes in their runs;
2. diagnose an Agent whose instruction names a tool that it cannot actually use;
3. repair a vague tool definition and inspect the repaired run; and
4. use callbacks as runtime guardrails, then optionally run one small regression eval.


## 1. Setup

Select this repository's `.venv` as the recommended notebook kernel. Then run the notebook cells from top to bottom, one at a time. Do not use **Run All**: several cells make live model requests, and the pauses for ADK Web inspection are part of the exercise.

Keep the kernel running while you inspect the sessions in ADK Web. Use the URL printed by the runtime-start cell.

The API key stays in your local environment. Never paste it or raw trace data into the public GitHub Issue. Free-tier limits vary by project and model; check your current limits in [Google AI Studio](https://ai.dev/rate-limit).


In [ ]:
from pathlib import Path
from typing import Literal
import hashlib
import json
import os
import sys
import warnings

from dotenv import load_dotenv
from google.adk.agents import Agent
from google.genai import types
from IPython.display import Markdown, display


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "spooky/data/glossary.snapshot.json").is_file() and (
            candidate / "README.md"
        ).is_file():
            return candidate
    raise RuntimeError("Run this Notebook from inside the Search Agent Lab repository.")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
EXPECTED_ENVIRONMENT = (REPOSITORY_ROOT / ".venv").resolve()
ACTIVE_ENVIRONMENT = Path(sys.prefix).resolve()
if ACTIVE_ENVIRONMENT == EXPECTED_ENVIRONMENT:
    print("Environment: repository .venv")
else:
    warnings.warn(
        "The repository .venv is recommended, but this notebook will continue "
        f"with the active environment: {ACTIVE_ENVIRONMENT}",
        RuntimeWarning,
        stacklevel=2,
    )
    print("Environment:", ACTIVE_ENVIRONMENT)

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

for env_path in (REPOSITORY_ROOT / ".env", EXPECTED_ENVIRONMENT.parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path, override=False)
        break

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError(
        "GOOGLE_API_KEY is not configured. Add it to the untracked repository .env."
    )

print("Credential: detected locally")


In [ ]:
from search_agent_lab.week2_runtime import Week2NotebookRuntime, show_run

previous_runtime = globals().get("week2_runtime")
if callable(getattr(previous_runtime, "stop", None)):
    previous_runtime.stop()

week2_runtime = Week2NotebookRuntime(REPOSITORY_ROOT)
week2_runtime.start()
MODEL = "gemini-3.5-flash"
TEMPERATURE = 0
ADK_WEB_URL = week2_runtime.base_url

print("ADK Web:", ADK_WEB_URL)


## 2. Compare three Agent configurations

### First: an Agent without glossary tools

Run the Agent below. It has no custom instruction and no glossary tools. It answers this question:

> What is the difference between a Tool and a Skill?

Run the next cell and note the printed app name and session ID.

The model may already know general information about tools and skills. A good answer does not prove that it used this course's definitions. We will compare the runs after all three are complete.


In [ ]:
COMPARISON_QUESTION = "What is the difference between a Tool and a Skill?"
BASELINE_APP = "week2_baseline_fluent"
baseline_agent = Agent(
    name=BASELINE_APP,
    model=MODEL,
    generate_content_config=types.GenerateContentConfig(temperature=TEMPERATURE),
)
week2_runtime.register_agent(BASELINE_APP, baseline_agent)

baseline_run = week2_runtime.run_agent(BASELINE_APP, COMPARISON_QUESTION)
show_run(baseline_run)


### Define the glossary tools

Now make an Agent answer the same question with the study-group glossary available. First define the two tools that it can use.

The study-group website builds its glossary data from human-edited Markdown files. This notebook uses a repository-local copy pinned to a recorded source commit, so every learner runs against the same definitions even if the public glossary later changes.

The next cell checks that snapshot and defines the two glossary tools in this notebook. They are ordinary Python functions: their names, type annotations, and docstrings are the information ADK uses to expose them to an Agent. The production Agent uses the same tool contract; its implementation lives in `spooky.tools` so the application can reuse it outside this notebook.


In [ ]:
from copy import deepcopy

CANONICAL_GLOSSARY_URL = (
    "https://absurdwall.github.io/search-agent-study-group/glossary/"
)

SNAPSHOT_PATH = REPOSITORY_ROOT / "spooky/data/glossary.snapshot.json"
META_PATH = REPOSITORY_ROOT / "spooky/data/glossary.snapshot.meta.json"
raw_snapshot = SNAPSHOT_PATH.read_bytes()
snapshot_metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
glossary_payload = json.loads(raw_snapshot)

assert hashlib.sha256(raw_snapshot).hexdigest() == snapshot_metadata["sha256"]
assert glossary_payload["schema_version"] == snapshot_metadata["schema_version"]
assert len(glossary_payload["terms"]) == snapshot_metadata["term_count"]

GLOSSARY_TERMS = tuple(glossary_payload["terms"])
GLOSSARY_BY_ID = {term["id"]: term for term in GLOSSARY_TERMS}


def _query_tokens(text: str) -> set[str]:
    """Normalize a short learner question into comparable words."""
    stop_words = {"a", "an", "and", "between", "difference", "is", "the", "what"}
    words = {word.strip(".,?!:;()[]{}\"'").casefold() for word in text.split()} - {""}
    return {word[:-1] if len(word) > 3 and word.endswith("s") else word for word in words - stop_words}


def search_glossary(query: str) -> dict[str, object]:
    """Search the local study-group glossary and return up to five candidate summaries."""
    if not isinstance(query, str):
        return {"status": "not_found", "query": "", "results": []}

    query_words = _query_tokens(query)
    matches = []
    for position, term in enumerate(GLOSSARY_TERMS):
        searchable_text = " ".join(
            [
                term["id"],
                term["term"],
                *term.get("aliases", []),
                term["sections"]["Simple definition"],
            ]
        )
        overlap = query_words & _query_tokens(searchable_text)
        name_words = _query_tokens(" ".join([term["id"], term["term"], *term.get("aliases", [])]))
        name_overlap = query_words & name_words
        if overlap:
            matches.append((10 * len(name_overlap) + len(overlap), position, term))

    matches.sort(key=lambda match: (-match[0], match[1]))
    results = [
        {
            "id": term["id"],
            "term": term["term"],
            "simple_definition": term["sections"]["Simple definition"],
        }
        for _, _, term in matches[:5]
    ]
    return {"status": "ok" if results else "not_found", "query": query, "results": results}


def get_glossary_terms(term_ids: list[str]) -> dict[str, object]:
    """Return complete local glossary records for requested canonical IDs."""
    requested_ids = term_ids if isinstance(term_ids, list) else []
    found = []
    missing_ids = []
    for term_id in requested_ids:
        normalized_id = term_id.casefold() if isinstance(term_id, str) else ""
        source = GLOSSARY_BY_ID.get(normalized_id)
        if source is None:
            missing_ids.append(normalized_id)
            continue
        record = deepcopy(source)
        record["canonical_url"] = f"{CANONICAL_GLOSSARY_URL}#{normalized_id}"
        found.append(record)

    status = "partial" if found and missing_ids else "ok" if found else "not_found"
    return {"status": status, "terms": found, "missing_ids": missing_ids}


print("Snapshot checksum: PASS")
print("Defined tools: search_glossary(query) and get_glossary_terms(term_ids)")
print("Glossary terms loaded:", len(GLOSSARY_TERMS))

GLOSSARY_INSTRUCTION = """You are a concise glossary guide for study-group learners. Search the study-group glossary before answering; identify relevant IDs; retrieve full records before substantive claims; use only returned glossary content; and link every concept to its canonical glossary page. For the usual flow, call search_glossary once with the complete question, then get_glossary_terms once with the relevant IDs."""


### Same instruction, no registered tools

Run this Agent before the glossary-backed Agent below. It uses the same GLOSSARY_INSTRUCTION and the same question, but tools=[].

In ADK Web, compare its Request and `finish_reason` with the later grounded run. Naming tools in an instruction does not register them.


In [ ]:
MISSING_TOOLS_APP = "week2_instruction_without_tools"
missing_tools_agent = Agent(
    name=MISSING_TOOLS_APP,
    model=MODEL,
    instruction=GLOSSARY_INSTRUCTION,
    tools=[],
    generate_content_config=types.GenerateContentConfig(temperature=TEMPERATURE),
)
week2_runtime.register_agent(MISSING_TOOLS_APP, missing_tools_agent)


In [ ]:
missing_tools_run = week2_runtime.run_agent(MISSING_TOOLS_APP, COMPARISON_QUESTION)
show_run(missing_tools_run)
print("In ADK Web, inspect the final model Response for its finish_reason.")


### Then register the glossary tools

The next Agent keeps the same instruction and question, but registers search_glossary and get_glossary_terms. This isolates the effect of tools=[...].


In [ ]:
GROUNDED_APP = "week2_grounded_glossary"
grounded_glossary_agent = Agent(
    name=GROUNDED_APP,
    model=MODEL,
    instruction=GLOSSARY_INSTRUCTION,
    tools=[search_glossary, get_glossary_terms],
    generate_content_config=types.GenerateContentConfig(temperature=TEMPERATURE),
)
week2_runtime.register_agent(GROUNDED_APP, grounded_glossary_agent)

grounded_glossary_run = week2_runtime.run_agent(
    GROUNDED_APP,
    COMPARISON_QUESTION,
)
show_run(grounded_glossary_run)


### Compare the three completed runs in ADK Web

Open the ADK Web URL printed in Setup. Compare the baseline run, the same-instruction tools=[] run, and the glossary-backed run.

1. Compare reply content.
2. Compare Events and Traces. The baseline has no custom tool path; the tools=[] run has no registered glossary schemas.
3. In the `tools=[]` run, open the final model Response and record its exact `finish_reason` for the checkpoint.

Use these three runs to separate model knowledge, instruction, and registered capability.


## 3. Assignment — improve a vague tool

Now add one tool for the Week 1 coverage question:

> Which glossary concepts were introduced in Week 1?

This Agent needs a different instruction from Section 2: the earlier instruction explicitly describes the search_glossary -> get_glossary_terms workflow. Here we use one purpose-specific instruction for both the vague and repaired runs. It explains the goal without naming the function or supplying its allowed argument values.

A vague tool contract can make the model guess repeatedly after an unhelpful result. To keep this experiment bounded, we will add runtime guardrails before running it.


### Callback and guardrail

A **callback** is a place where ADK calls our Python code at a specific moment in the Agent loop. It is a mechanism, not a safety rule by itself.

A **guardrail** is a rule that constrains what the Agent may do. Here we implement two guardrails with callbacks:

- `before_model_callback` allows at most three real model requests for one learner question. Returning an `LlmResponse` skips the next API request and ends the loop with a clear message.
- `before_tool_callback` allows the registered week lookup tool to execute at most once for that question. Later attempts receive a blocked result without running the Python function again.

Without these guardrails, the model may repeatedly guess new inputs after `not_found`, consuming requests until the client timeout or API quota stops it. The instruction remains identical between the vague and repaired runs, so the learner can isolate the effect of the tool contract.


In [ ]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools import BaseTool, ToolContext

MAX_MODEL_CALLS_PER_TURN = 3
MAX_WEEK_TOOL_CALLS_PER_TURN = 1

def stop_runaway_model_loop(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> LlmResponse | None:
    key = f"temp:model_calls:{callback_context.invocation_id}"
    count = callback_context.state.get(key, 0) + 1
    callback_context.state[key] = count
    if count <= MAX_MODEL_CALLS_PER_TURN:
        return None
    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[types.Part(text="Stopped: this question reached the model-call guardrail.")],
        )
    )

def limit_week_tool_calls(
    tool: BaseTool,
    args: dict[str, object],
    tool_context: ToolContext,
) -> dict[str, object] | None:
    key = f"temp:week_tool_calls:{tool_context.invocation_id}"
    count = tool_context.state.get(key, 0)
    if count >= MAX_WEEK_TOOL_CALLS_PER_TURN:
        return {"status": "blocked", "reason": "The week lookup tool may run only once per question."}
    tool_context.state[key] = count + 1
    return None


In [ ]:
WEEK_QUESTION = "Which glossary concepts were introduced in Week 1?"

WEEK_COVERAGE_INSTRUCTION = """You are a concise glossary guide for study-group learners.
When a learner asks which concepts were introduced in a study-group week, use the available glossary tool.
Base the answer only on returned glossary records and include their canonical links."""

def find_terms_by_week(introduced_in: str) -> dict[str, object]:
    """Some function."""
    terms = [
        {
            "id": term["id"],
            "term": term["term"],
            "canonical_url": f"{CANONICAL_GLOSSARY_URL}#{term['id']}",
        }
        for term in GLOSSARY_TERMS
        if term.get("introduced_in") == introduced_in
    ]
    return {
        "status": "ok" if terms else "not_found",
        "introduced_in": introduced_in,
        "count": len(terms),
        "terms": terms,
    }

VAGUE_APP = "week2_vague_week_lookup"
vague_agent = Agent(
    name=VAGUE_APP,
    model=MODEL,
    instruction=WEEK_COVERAGE_INSTRUCTION,
    tools=[find_terms_by_week],
    before_model_callback=stop_runaway_model_loop,
    before_tool_callback=limit_week_tool_calls,
    generate_content_config=types.GenerateContentConfig(temperature=TEMPERATURE),
)
week2_runtime.register_agent(VAGUE_APP, vague_agent)
vague_run = week2_runtime.run_agent(VAGUE_APP, WEEK_QUESTION)
show_run(vague_run)


### Inspect what the LLM receives

The model does not receive the Python function body. ADK sends only the function declaration: its name, description, parameters, and required fields. Run the next cell before repairing the tool.

Your trace may still show a second find_terms_by_week event. That event records the model's second tool attempt. Open its functionResponse: the before_tool_callback should return status=blocked, and ADK should skip the Python function body. A tool event therefore does not always mean that the underlying function executed.


In [ ]:
from google.adk.tools import FunctionTool

vague_declaration = FunctionTool(find_terms_by_week)._get_declaration()
print(json.dumps(vague_declaration.model_dump(exclude_none=True, mode="json"), indent=2, sort_keys=True))


### Repair the tool contract

The function is already named `find_terms_by_week`, and its lookup body is already complete. Use the declaration above as evidence and improve only the interface information that remains vague:

- improve the docstring; and
- improve the definition of `introduced_in`.

The checker focuses on that tool contract and runs one smoke test to confirm that the provided lookup behavior still works. After it passes, rerun the Agent with the same `WEEK_COVERAGE_INSTRUCTION`. Keeping the name, function body, and instruction fixed makes the comparison focus on the repaired contract.


In [ ]:
def find_terms_by_week(introduced_in: str) -> dict[str, object]:
    """Some function."""
    terms = [{"id": t["id"], "term": t["term"], "canonical_url": f"{CANONICAL_GLOSSARY_URL}#{t['id']}"} for t in GLOSSARY_TERMS if t.get("introduced_in") == introduced_in]
    return {"status": "ok" if terms else "not_found", "introduced_in": introduced_in, "count": len(terms), "terms": terms}

# TODO: Improve the docstring and the definition of introduced_in.


In [ ]:
from google.adk.tools import FunctionTool

def check_week_tool(function):
    try:
        declaration = FunctionTool(function)._get_declaration().model_dump(exclude_none=True, mode="json")
        properties = declaration["parameters_json_schema"].get("properties", {})
        expected_terms = [
            {"id": term["id"], "term": term["term"], "canonical_url": f"{CANONICAL_GLOSSARY_URL}#{term['id']}"}
            for term in GLOSSARY_TERMS if term.get("introduced_in") == "week-01"
        ]
        result = function("week-01")
        description = str(declaration.get("description", "")).casefold()
        checks = {
            "docstring describes a glossary week lookup": "glossary" in description and "week" in description,
            "only introduced_in is exposed": tuple(properties) == ("introduced_in",),
            "introduced_in lists the two valid labels": properties.get("introduced_in", {}).get("enum") == ["week-01", "week-02"],
            "introduced_in is required": declaration["parameters_json_schema"].get("required") == ["introduced_in"],
            "provided Week 1 behavior still works": result == {"status": "ok", "introduced_in": "week-01", "count": len(expected_terms), "terms": expected_terms},
        }
        hints = {
            "docstring describes a glossary week lookup": "Describe what glossary concepts this tool returns and how the week is selected.",
            "introduced_in lists the two valid labels": 'Hint: annotate introduced_in with Literal["week-01", "week-02"].',
        }
    except Exception as error:
        checks = {"ADK declaration and function call": False}
        hints = {}
        print("Checker error:", type(error).__name__)
    for label, passed in checks.items():
        print("[PASS]" if passed else "[FAIL]", label)
        if not passed and label in hints:
            print("       " + hints[label])
    return {"passed": all(checks.values()), "checks": checks}

contract_report = check_week_tool(find_terms_by_week)
print("Contract checker:", "PASS" if contract_report["passed"] else "NOT READY")


### Run the repaired Agent

The next Agent registers your repaired find_terms_by_week. In ADK Web, compare this run with the vague run: the call should now use introduced_in=week-01 and return the seven Week 1 terms.


In [ ]:
repaired_week_run = None
if not contract_report["passed"]:
    print("Repaired Agent not started. Fix the contract failures, then rerun from your function cell.")
else:
    REPAIRED_WEEK_APP = "week2_repaired_week_lookup"
    repaired_week_agent = Agent(
        name=REPAIRED_WEEK_APP,
        model=MODEL,
        instruction=WEEK_COVERAGE_INSTRUCTION,
        tools=[find_terms_by_week],
        before_model_callback=stop_runaway_model_loop,
        before_tool_callback=limit_week_tool_calls,
        generate_content_config=types.GenerateContentConfig(temperature=TEMPERATURE),
    )
    week2_runtime.register_agent(REPAIRED_WEEK_APP, repaired_week_agent)
    repaired_week_run = week2_runtime.run_agent(REPAIRED_WEEK_APP, WEEK_QUESTION)
    show_run(repaired_week_run)


Open the printed `week2_repaired_week_lookup` session in ADK Web:

1. Request contains `find_terms_by_week`.
2. The model sends a `functionCall` with `introduced_in=week-01`.
3. Its `functionResponse` contains the seven Week 1 terms and canonical links.
4. The final answer comes after that result.


### Submit the Week 2 checkpoint

Fill in the three values in the next cell.

**Field 1**

Enter the exact `finish_reason` from the `week2_instruction_without_tools` session's final model **Response**.

**Field 2**

Match each observation to the component that directly causes it. Use:

- **I** = Agent instruction
- **T** = registered tool
- **L** = LLM
- **C** = callback guardrail

1. Produces a structured `functionCall` requesting `find_terms_by_week`.
2. Tells the Agent to use a glossary tool and include canonical links.
3. Returns `status=blocked` for a second tool attempt without running the lookup again.
4. Registering it exposes the tool name, description, parameter schema, and callable capability.

Enter the four letters in order as one string.

**Field 3**

Enter your GitHub username.

Run the validation cell. If an answer is wrong, the notebook tells you where to check. When all checks pass, open the generated Issue link and submit it.

Do not add your API key, raw trace, prompt, response, or local environment data to the public Issue.


In [ ]:
FINISH_REASON_ANSWER = ""
# Intentionally wrong example: replace it with your four-letter answer.
COMPONENT_MATCHES = "TLIC"
GITHUB_USERNAME = ""


In [ ]:
from search_agent_lab.checkpoints import (
    WEEK_02,
    build_issue_form_url,
    expected_evidence,
    generate_codename,
    normalize_github_username,
)
from search_agent_lab.week2_contract import assess_week2_checkpoint_answers

answer_assessment = assess_week2_checkpoint_answers(
    FINISH_REASON_ANSWER,
    COMPONENT_MATCHES,
)

if not contract_report["passed"]:
    display(Markdown("**Coding checkpoint not ready.** Return to the contract checker."))
if not answer_assessment.finish_reason_passed:
    display(Markdown("**Field 1 needs another look.** Open the missing-capability session, select the model event, and inspect Response metadata rather than the HTTP status."))
if not answer_assessment.component_matches_passed:
    display(Markdown("**Field 2 needs another look.** For each observation, identify which component directly produced, instructed, blocked, or exposed it."))
if repaired_week_run is None:
    display(Markdown("**Live run not ready.** Return to the repaired Agent cell."))

if contract_report["passed"] and answer_assessment.passed and repaired_week_run is not None:
    try:
        normalized_username = normalize_github_username(GITHUB_USERNAME)
    except ValueError as exc:
        display(Markdown(f"**GitHub username:** {exc}"))
    else:
        evidence = expected_evidence(WEEK_02)
        codename = generate_codename(normalized_username, WEEK_02, evidence)
        issue_url = build_issue_form_url(normalized_username, WEEK_02, evidence)
        display(Markdown(
            f"### Checkpoint passed\n\n**{codename}**\n\n"
            f"[Open the prefilled GitHub Issue]({issue_url})"
        ))


### Optional: turn the repaired behavior into an eval

The required checkpoint is complete. This optional extension reruns one saved case to check whether a future Agent change preserves the expected tool path and produces an acceptable answer. An eval detects regressions; it does not protect a live run or stop a loop.

Evaluation support comes from the `google-adk[eval]` dependency included in this repository's `requirements.lock`. If ADK Web says the eval module is not installed, stop the notebook server, activate the repository `.venv`, run `python -m pip install --upgrade pip` followed by `python -m pip install --require-hashes -r requirements.lock`, and restart the kernel before continuing.

ADK Web discovers eval files from a directory named after the app. The repaired app's saved set is `week2_repaired_week_lookup/week1_coverage.evalset.json`. Run the next cell to inspect that saved contract before executing it.


In [ ]:
EVAL_SET_PATH = REPOSITORY_ROOT / "week2_repaired_week_lookup/week1_coverage.evalset.json"
eval_set_payload = json.loads(EVAL_SET_PATH.read_text(encoding="utf-8"))
eval_case = eval_set_payload["eval_cases"][0]
expected_invocation = eval_case["conversation"][0]

eval_contract = {
    "question": expected_invocation["userContent"]["parts"][0]["text"],
    "expected_tool_call": expected_invocation["intermediateData"]["toolUses"][0],
    "reference_answer": expected_invocation["finalResponse"]["parts"][0]["text"],
}
print("Eval file:", EVAL_SET_PATH.relative_to(REPOSITORY_ROOT))
print(json.dumps(eval_contract, indent=2))


### Run and interpret the eval in ADK Web

When you run this eval, ADK does more than reopen the saved trace:

1. It creates a new evaluation run for `week2_repaired_week_lookup`.
2. It sends the saved Week 1 question to the current Agent definition.
3. It records the new model calls, tool calls, tool responses, and final answer.
4. It compares the actual tool trajectory with the expected `find_terms_by_week(introduced_in="week-01")` call.
5. It compares the new final answer with the saved reference answer using the selected response metric.

In ADK Web:

1. Select the `week2_repaired_week_lookup` app and open **Evals**.
2. Open the `week1_coverage` eval set. It contains one case, `week1_concepts`.
3. Click `week1_concepts` if you want to inspect the saved question, expected tool call, and reference answer. This view does **not** run the Agent. Return to the eval-set list when you are done.
4. Click **Run All**. In **Evaluation Metrics**, keep **Tool Trajectory** and **Response Match** selected. For this notebook, leave their current thresholds at `1` and `0.7`, then click **Start**.
5. After the run, open its result under **Past Runs** and inspect the two metric results separately.

A failed run is labeled **Fail**. The metric details tell you why: **Tool Trajectory** fails when the actual tool name, arguments, or sequence does not match closely enough; **Response Match** fails when the final answer scores below its selected threshold. One metric can pass while the other fails.

This one-case eval costs one additional Agent run. Its purpose is to turn today's successful behavior into a regression check for future changes.


## 4. Summary and cleanup

- An instruction can name a function, but the Agent cannot use it until you register it in `tools`.
- ADK builds a tool definition from a Python function's name, docstring, type hints, and defaults.
- The lookup algorithm can stay unchanged while the Agent-facing tool contract becomes clearer.
- `functionCall` is the model's request to use a tool; `functionResponse` is the result returned through ADK.
- A callback is Python code that ADK runs at a defined point in the Agent loop.
- A guardrail is a runtime rule; callbacks are one way to enforce it.
- An optional eval reruns saved cases to detect regressions; it does not enforce runtime limits.
- ADK Web shows each step from the request to the final answer.

### Cleanup

Keep the server running until you finish the ADK Web inspection and, if you chose it, the optional eval. Then set `STOP_SERVER = True` and run the last cell. Stopping the server removes its in-memory sessions.


In [ ]:
STOP_SERVER = False

if STOP_SERVER:
    week2_runtime.stop()
    print("Notebook ADK Web stopped; in-memory sessions removed.")
else:
    print("Notebook ADK Web remains available at", ADK_WEB_URL)
